In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import RBFInterpolator
import sys, os

cwd = os.getcwd()
root_path = os.path.abspath(os.path.join(cwd, os.pardir))
sys.path.append(root_path)

## Notes Example 1 — Planar Data

Data: `(0,0,0), (1,0,1), (0,1,1), (1,1,2)` lies on the plane `z = x + y`.
TPS reproduces polynomials of degree ≤ 1 exactly, so all weights should be zero.

In [ ]:
from implementation.thin_plate_spline_interpolation import thin_plate_spline_interpolation

x1 = np.array([0.0, 1.0, 0.0, 1.0])
y1 = np.array([0.0, 0.0, 1.0, 1.0])
z1 = np.array([0.0, 1.0, 1.0, 2.0])

test_pts = [(0.5, 0.5), (0.25, 0.75), (0.3, 0.7)]
print('Example 1 — Planar data (z = x + y)')
print(f'{"Point":>15}  {"TPS":>10}  {"Exact":>10}')
for pt in test_pts:
    val = thin_plate_spline_interpolation(x1, y1, z1, pt)
    exact = pt[0] + pt[1]
    print(f'{str(pt):>15}  {val:10.6f}  {exact:10.6f}')

## Notes Example 2 — Non-Planar Saddle Data

Data: `(0,0,1), (1,0,0), (0,1,0), (1,1,1)` — by symmetry, `f(0.5, 0.5) = 0.5`.

In [ ]:
x2 = np.array([0.0, 1.0, 0.0, 1.0])
y2 = np.array([0.0, 0.0, 1.0, 1.0])
z2 = np.array([1.0, 0.0, 0.0, 1.0])

coords = np.column_stack([x2, y2])
rbf = RBFInterpolator(coords, z2, kernel='thin_plate_spline', degree=1)

print('Example 2 — Non-planar data')
print(f'{"Point":>15}  {"TPS":>10}  {"scipy":>10}  {"Match?":>8}')
for pt in [(0.5,0.5), (0.25,0.75), (0.3,0.1)]:
    val = thin_plate_spline_interpolation(x2, y2, z2, pt)
    ref = float(rbf(np.array([pt]))[0])
    print(f'{str(pt):>15}  {val:10.6f}  {ref:10.6f}  {np.isclose(val, ref):>8}')

## Scattered sin(x)·cos(y) — Implementation vs scipy Surface Plots

In [ ]:
np.random.seed(42)
n_pts = 25
xs = np.random.uniform(0, 3, n_pts)
ys = np.random.uniform(0, 3, n_pts)
zs = np.sin(xs) * np.cos(ys)

grid_n = 30
gx = np.linspace(0, 3, grid_n)
gy = np.linspace(0, 3, grid_n)
GX, GY = np.meshgrid(gx, gy)

GZ_impl = np.zeros_like(GX)
for i in range(grid_n):
    for j in range(grid_n):
        GZ_impl[i, j] = thin_plate_spline_interpolation(xs, ys, zs, (GX[i,j], GY[i,j]))

rbf2 = RBFInterpolator(np.column_stack([xs, ys]), zs, kernel='thin_plate_spline', degree=1)
GZ_scipy = rbf2(np.column_stack([GX.ravel(), GY.ravel()])).reshape(GX.shape)

fig = plt.figure(figsize=(16, 5))

ax1 = fig.add_subplot(131, projection='3d')
ax1.plot_surface(GX, GY, GZ_impl, cmap='viridis', alpha=0.8)
ax1.scatter(xs, ys, zs, color='red', s=40)
ax1.set_title('Implementation')

ax2 = fig.add_subplot(132, projection='3d')
ax2.plot_surface(GX, GY, GZ_scipy, cmap='viridis', alpha=0.8)
ax2.scatter(xs, ys, zs, color='red', s=40)
ax2.set_title('scipy RBFInterpolator')

ax3 = fig.add_subplot(133, projection='3d')
GZ_exact = np.sin(GX) * np.cos(GY)
ax3.plot_surface(GX, GY, GZ_exact, cmap='viridis', alpha=0.8)
ax3.scatter(xs, ys, zs, color='red', s=40)
ax3.set_title('Exact: sin(x)·cos(y)')

plt.suptitle('TPS Interpolation: Implementation vs scipy vs Exact')
plt.tight_layout()
plt.show()

print(f'Max |impl − scipy| = {np.abs(GZ_impl - GZ_scipy).max():.2e}')